# Skin Lesion Classification with EfficientNet-B0

This notebook presents a supervised image classification pipeline for skin lesion diagnosis using the HAM10000 dataset.

A pretrained EfficientNet-B0 model initialized with ImageNet weights is fine-tuned for multi-class lesion classification. Training follows a two-stage procedure: the classification head is trained while the backbone remains frozen, after which the final layers of the backbone are unfrozen for fine-tuning with a reduced learning rate.

To prevent information leakage, the dataset is partitioned using lesion-level stratification with `StratifiedGroupKFold`, ensuring that images from the same lesion do not appear in multiple splits.

Given the pronounced class imbalance in HAM10000, class-weighted loss is used during optimization. Data augmentation is performed online using Keras preprocessing layers, and images are streamed from disk through the `tf.data` pipeline for efficient training.

Model performance is evaluated using macro F1 score, balanced accuracy, and macro ROC-AUC, which provide more informative assessments than overall accuracy on imbalanced datasets.


In [ ]:
# Kaggle environment detection & setup
import os as _os, pathlib as _pl, glob as _glob, sys as _sys
KAGGLE_DATASET_SLUG = 'skin-cancer-mnist-ham10000'
KAGGLE_WORKING = '/kaggle/working'
ON_KAGGLE = _pl.Path('/kaggle/input').exists()
if ON_KAGGLE:
    print('Running on Kaggle')
    _os.makedirs(KAGGLE_WORKING, exist_ok=True)
    _os.chdir(KAGGLE_WORKING)
    hits = _glob.glob(f'/kaggle/input/**/{KAGGLE_DATASET_SLUG}', recursive=True)
    _kaggle_path = hits[0] if hits else '/kaggle/input'
    print(f'Dataset: {_kaggle_path}')
    import kagglehub as _kh
    _kh.dataset_download = lambda h, *a, **kw: _kaggle_path
    import builtins as _bi
    _real_open = _bi.open
    _bi.open = _real_open
    _os.environ['RESULTS_BASE'] = KAGGLE_WORKING
print('Setup complete')


Running on Kaggle
Dataset: /kaggle/input
Setup complete


### Imports

Import TensorFlow, scikit-learn, and visualization libraries.


In [ ]:
import os, time, json, random, zipfile, platform, pathlib
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    recall_score, precision_score, f1_score,
    balanced_accuracy_score, roc_auc_score,
)

from tensorflow import keras
from tensorflow.keras import layers


### Configuration

Define hyperparameters, paths, random seed, and output directory.


In [ ]:
SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 16
HEAD_EPOCHS = 15
FINE_TUNE_EPOCHS = 15
FINE_TUNE_LAYERS = 30
HEAD_LR = 1e-3
FINE_TUNE_LR = 1e-5
MODEL_NAME = "EfficientNetB0"
DATASET_NAME = "HAM10000"
DATASET_HANDLE = "kmader/skin-cancer-mnist-ham10000"

EXP_NAME = "ham10000_b0"
RESULTS_DIR = Path(_os.environ.get("RESULTS_BASE", "/content/results")) / EXP_NAME
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Python: {platform.python_version()}")
print(f"TensorFlow: {tf.__version__}")
print(f"GPU devices: {tf.config.list_physical_devices('GPU')}")

tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120


Python: 3.12.13
TensorFlow: 2.20.0
GPU devices: []


### Data Download

Download and cache the HAM10000 dataset using `kagglehub`.


In [ ]:
print(f"Downloading dataset: {DATASET_HANDLE}")
data_path = kagglehub.dataset_download(DATASET_HANDLE)
print(f"Dataset directory: {data_path}")

metadata_path = Path(data_path) / "HAM10000_metadata.csv"
df = pd.read_csv(metadata_path)
print(f"Metadata shape: {df.shape}")


Dataset directory: /kaggle/input


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/HAM10000_metadata.csv'

### Image Indexing and Label Encoding

Map image IDs to file paths and encode diagnosis labels.


In [ ]:
part1_dir = Path(data_path) / "HAM10000_images_part_1"
part2_dir = Path(data_path) / "HAM10000_images_part_2"

for image_dir, archive_name in [
    (part1_dir, "HAM10000_images_part_1.zip"),
    (part2_dir, "HAM10000_images_part_2.zip"),
]:
    if not image_dir.is_dir():
        archive_path = Path(data_path) / archive_name
        if not archive_path.exists():
            raise FileNotFoundError(f"Neither {image_dir} nor {archive_path} was found.")
        print(f"Extracting {archive_name}...")
        with zipfile.ZipFile(archive_path, "r") as archive:
            archive.extractall(data_path)

path_lookup = {
    image_path.stem: str(image_path)
    for image_dir in (part1_dir, part2_dir)
    for image_path in image_dir.glob("*.jpg")
}

df["image_path"] = df["image_id"].map(path_lookup)
missing_paths = df["image_path"].isna().sum()
if missing_paths:
    raise ValueError(f"{missing_paths} metadata rows have no matching image file.")

label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["dx"])
class_names = label_encoder.classes_
num_classes = len(class_names)
print(f"Indexed images: {len(df):,}")
print(f"Classes ({num_classes}): {list(class_names)}")

np.save(str(RESULTS_DIR / "class_names.npy"), class_names)


### Class Distribution

Visualize class frequencies to assess dataset imbalance.


In [ ]:
class_distribution = (
    df["dx"].value_counts().reindex(class_names).rename_axis("class").reset_index(name="count")
)
class_distribution["percent"] = (
    class_distribution["count"] / class_distribution["count"].sum() * 100
)
print("Class distribution:")
print(class_distribution.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=class_distribution, x="class", y="count", color="#3b82f6", ax=ax)
ax.set(title=f"{DATASET_NAME} diagnosis distribution", xlabel="Diagnosis", ylabel="Image count")
fig.tight_layout()
fig.savefig(str(RESULTS_DIR / "class_distribution.png"), dpi=150)
plt.close(fig)


### Lesion-Disjoint Stratified Split

Create lesion-disjoint train, validation, and test splits with `StratifiedGroupKFold`.


In [ ]:
labels = df["label"].to_numpy()
groups = df["lesion_id"].to_numpy()
all_indices = np.arange(len(df))

outer_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_val_idx, test_idx = next(outer_splitter.split(all_indices, y=labels, groups=groups))

train_val_df = df.iloc[train_val_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

inner_splitter = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
train_idx, val_idx = next(
    inner_splitter.split(
        train_val_df.index,
        y=train_val_df["label"],
        groups=train_val_df["lesion_id"],
    )
)

train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
val_df = train_val_df.iloc[val_idx].reset_index(drop=True)

paths_train = train_df["image_path"].to_numpy()
y_train = train_df["label"].to_numpy()
paths_val = val_df["image_path"].to_numpy()
y_val = val_df["label"].to_numpy()
paths_test = test_df["image_path"].to_numpy()
y_test = test_df["label"].to_numpy()

train_lesions = set(train_df["lesion_id"])
val_lesions = set(val_df["lesion_id"])
test_lesions = set(test_df["lesion_id"])
assert train_lesions.isdisjoint(val_lesions)
assert train_lesions.isdisjoint(test_lesions)
assert val_lesions.isdisjoint(test_lesions)

split_sizes = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "images": [len(train_df), len(val_df), len(test_df)],
    "lesions": [train_df["lesion_id"].nunique(), val_df["lesion_id"].nunique(), test_df["lesion_id"].nunique()],
})
split_sizes["image_percent"] = split_sizes["images"] / split_sizes["images"].sum() * 100
print("\nSplit sizes:")
print(split_sizes.to_string(index=False))


### Streaming tf.data Pipeline

Build an efficient streaming input pipeline with on-the-fly preprocessing.


In [ ]:
def decode_and_resize(image_path, label):
    image_bytes = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image_bytes, channels=3)
    image = tf.image.resize(image, (IMAGE_SIZE, IMAGE_SIZE), antialias=True)
    image = tf.cast(image, tf.float32)
    return image, label

def make_dataset(paths, labels, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        dataset = dataset.shuffle(buffer_size=min(len(paths), 2048), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(decode_and_resize, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return dataset

train_ds = make_dataset(paths_train, y_train, training=True)
val_ds = make_dataset(paths_val, y_val)
test_ds = make_dataset(paths_test, y_test)

sample_images, sample_labels = next(iter(train_ds))
print(f"\nBatch image shape: {sample_images.shape}")
print(f"Batch label shape: {sample_labels.shape}")
print(f"Pixel range: {float(tf.reduce_min(sample_images)):.1f} to {float(tf.reduce_max(sample_images)):.1f}")


### Class Weighting

Compute balanced class weights from the training set.


In [ ]:
class_weights = compute_class_weight(class_weight="balanced", classes=np.arange(num_classes), y=y_train)
class_weight_dict = dict(zip(range(num_classes), class_weights))

weight_table = pd.DataFrame({
    "class": class_names,
    "train_images": [(y_train == i).sum() for i in range(num_classes)],
    "class_weight": class_weights,
})
print("\nClass weights:")
print(weight_table.to_string(index=False))


### Model Architecture

Construct an EfficientNet-B0 transfer learning model with online augmentation.


In [ ]:
augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.10),
        layers.RandomZoom(0.10),
        layers.RandomContrast(0.10),
    ],
    name="augmentation",
)

base_model = keras.applications.EfficientNetB0(
    include_top=False, weights="imagenet", input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)
)
base_model.trainable = False

inputs = keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3), name="image")
x = augmentation(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
x = layers.Dropout(0.30, name="dropout")(x)
outputs = layers.Dense(num_classes, activation="softmax", name="diagnosis")(x)

model = keras.Model(inputs=inputs, outputs=outputs, name=f"{MODEL_NAME}_{DATASET_NAME}")
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=HEAD_LR),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)
model.summary()


### Stage 1: Head Training

Train the classification head with the backbone frozen.


In [ ]:
head_callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2, min_lr=1e-6),
    keras.callbacks.ModelCheckpoint(
        str(RESULTS_DIR / "head_best.keras"), monitor="val_loss", save_best_only=True
    ),
]

t0 = time.perf_counter()
history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=HEAD_EPOCHS,
    class_weight=class_weight_dict,
    callbacks=head_callbacks,
    verbose=1,
)
head_time = time.perf_counter() - t0
print(f"Stage 1 training time: {head_time:.1f}s")


### Stage 2: Fine-Tuning

Fine-tune the final backbone layers using a reduced learning rate.


In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-FINE_TUNE_LAYERS]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=FINE_TUNE_LR),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)

fine_tune_callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2, min_lr=1e-7),
    keras.callbacks.ModelCheckpoint(
        str(RESULTS_DIR / "finetuned_best.keras"), monitor="val_loss", save_best_only=True
    ),
]

history_fine_tune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINE_TUNE_EPOCHS,
    class_weight=class_weight_dict,
    callbacks=fine_tune_callbacks,
    verbose=1,
)
fine_tune_time = time.perf_counter() - t0 - head_time
total_time = head_time + fine_tune_time
print(f"Stage 2 training time: {fine_tune_time:.1f}s")
print(f"Total training time: {total_time:.1f}s")


### Save Epoch History

Export training history for both optimization stages.


In [ ]:
for stage, history in [("head", history_head), ("finetune", history_fine_tune)]:
    hist_df = pd.DataFrame(history.history)
    hist_df.index.name = "epoch"
    hist_df.to_csv(str(RESULTS_DIR / f"epoch_history_{stage}.csv"))
    print(f"Saved epoch history: epoch_history_{stage}.csv")


### Training Curves

Plot learning curves for loss and accuracy.


In [ ]:
def plot_history(history, title):
    history_frame = pd.DataFrame(history.history)
    epochs = np.arange(1, len(history_frame) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(epochs, history_frame["loss"], label="Train")
    axes[0].plot(epochs, history_frame["val_loss"], label="Validation")
    axes[0].set(title=f"{title} loss", xlabel="Epoch", ylabel="Loss")
    axes[0].legend()
    axes[1].plot(epochs, history_frame["accuracy"], label="Train")
    axes[1].plot(epochs, history_frame["val_accuracy"], label="Validation")
    axes[1].set(title=f"{title} accuracy", xlabel="Epoch", ylabel="Accuracy")
    axes[1].legend()
    fig.tight_layout()
    return fig

fig_head = plot_history(history_head, "Stage 1: frozen EfficientNetB0 base")
fig_head.savefig(str(RESULTS_DIR / "training_plots_head.png"), dpi=150)
plt.close(fig_head)

fig_ft = plot_history(history_fine_tune, "Stage 2: fine-tuned EfficientNetB0")
fig_ft.savefig(str(RESULTS_DIR / "training_plots_finetune.png"), dpi=150)
plt.close(fig_ft)

# Combined overlay
fig_all, axes_all = plt.subplots(1, 2, figsize=(13, 4))
for stage, hist, label in [
    ("head", history_head, "Head"),
    ("finetune", history_fine_tune, "Fine-tune"),
]:
    hf = pd.DataFrame(hist.history)
    ep = np.arange(1, len(hf) + 1)
    axes_all[0].plot(ep, hf["loss"], label=f"{label} train")
    axes_all[0].plot(ep, hf["val_loss"], label=f"{label} val")
    axes_all[1].plot(ep, hf["accuracy"], label=f"{label} train")
    axes_all[1].plot(ep, hf["val_accuracy"], label=f"{label} val")
axes_all[0].set(title="Combined loss", xlabel="Epoch", ylabel="Loss")
axes_all[0].legend()
axes_all[1].set(title="Combined accuracy", xlabel="Epoch", ylabel="Accuracy")
axes_all[1].legend()
fig_all.tight_layout()
fig_all.savefig(str(RESULTS_DIR / "training_plots.png"), dpi=150)
plt.close(fig_all)


### Test Evaluation

Evaluate model performance on the held-out test set.


In [ ]:
test_probabilities = model.predict(test_ds, verbose=0)
y_pred = test_probabilities.argmax(axis=1)

test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
macro_f1 = f1_score(y_test, y_pred, average="macro")
balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
per_class_precision = precision_score(y_test, y_pred, average=None, zero_division=0)
per_class_recall = recall_score(y_test, y_pred, average=None)
per_class_f1 = f1_score(y_test, y_pred, average=None, zero_division=0)

y_test_one_hot = label_binarize(y_test, classes=np.arange(num_classes))
macro_roc_auc = roc_auc_score(y_test_one_hot, test_probabilities, average="macro", multi_class="ovr")

print(f"\n{'='*60}")
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Balanced accuracy: {balanced_accuracy:.4f}")
print(f"Macro ROC-AUC (OvR): {macro_roc_auc:.4f}")
print(classification_report(y_test, y_pred, target_names=class_names, digits=4))


### Save Metrics

Export overall and per-class evaluation metrics.


In [ ]:
overall_metrics = pd.DataFrame({
    "metric": ["test_loss", "test_accuracy", "macro_f1", "balanced_accuracy", "macro_roc_auc"],
    "value": [test_loss, test_accuracy, macro_f1, balanced_accuracy, macro_roc_auc],
})
overall_metrics.to_csv(str(RESULTS_DIR / "overall_metrics.csv"), index=False)

per_class_metrics = pd.DataFrame({
    "class": class_names,
    "test_images": [(y_test == i).sum() for i in range(num_classes)],
    "precision": per_class_precision,
    "recall": per_class_recall,
    "f1": per_class_f1,
})
per_class_metrics.to_csv(str(RESULTS_DIR / "per_class_metrics.csv"), index=False)


### Confusion Matrices

Generate raw and normalized confusion matrices.


In [ ]:
confusion = confusion_matrix(y_test, y_pred)
confusion_df = pd.DataFrame(confusion, index=class_names, columns=class_names)
confusion_df.index.name = "actual"
confusion_df.to_csv(str(RESULTS_DIR / "confusion_raw.csv"))

normalized_confusion = confusion.astype(float) / confusion.sum(axis=1, keepdims=True)
norm_conf_df = pd.DataFrame(normalized_confusion, index=class_names, columns=class_names)
norm_conf_df.index.name = "actual"
norm_conf_df.to_csv(str(RESULTS_DIR / "confusion_normalized.csv"))

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sns.heatmap(confusion, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set(title=f"{MODEL_NAME} {DATASET_NAME}: test confusion matrix", xlabel="Predicted", ylabel="Actual")
sns.heatmap(normalized_confusion, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set(title=f"{MODEL_NAME} {DATASET_NAME}: normalized confusion matrix", xlabel="Predicted", ylabel="Actual")
fig.tight_layout()
fig.savefig(str(RESULTS_DIR / "confusion_matrix.png"), dpi=150)
plt.close(fig)


### Save Model

Save the trained model in Keras format.


In [ ]:
MODEL_PATH = str(RESULTS_DIR / "model_final.keras")
model.save(MODEL_PATH)
model_size_mb = Path(MODEL_PATH).stat().st_size / (1024 ** 2)
print(f"Saved model: {MODEL_PATH} ({model_size_mb:.2f} MB)")


### Inference Benchmark

Measure and record inference latency.


In [ ]:
benchmark_images, _ = next(iter(test_ds))
for _ in range(3):
    _ = model(benchmark_images, training=False)  # warm-up

timings = []
for _ in range(10):
    start = time.perf_counter()
    _ = model(benchmark_images, training=False)
    timings.append(time.perf_counter() - start)

median_ms_per_image = np.median(timings) / len(benchmark_images) * 1000

benchmark_table = pd.DataFrame([{
    "model": MODEL_NAME,
    "dataset": DATASET_NAME,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "split_protocol": "StratifiedGroupKFold by lesion_id",
    "parameters": model.count_params(),
    "num_classes": num_classes,
    "head_epochs": HEAD_EPOCHS,
    "fine_tune_epochs": FINE_TUNE_EPOCHS,
    "head_time_s": head_time,
    "fine_tune_time_s": fine_tune_time,
    "total_time_s": total_time,
    "test_accuracy": test_accuracy,
    "macro_f1": macro_f1,
    "balanced_accuracy": balanced_accuracy,
    "macro_roc_auc": macro_roc_auc,
    "saved_model_mb": model_size_mb,
    "median_inference_ms_per_image": median_ms_per_image,
}])
benchmark_table.to_csv(str(RESULTS_DIR / "benchmark.csv"), index=False)
print(f"Median inference: {median_ms_per_image:.2f} ms/image")


### Per-Class Recall Summary

Summarize recall for each lesion class.


In [ ]:
class_recall = dict(zip(class_names, per_class_recall))
for cls in class_names:
    print(f"  {cls} recall: {class_recall[cls]:.4f}")


### Experiment Summary

Generate a consolidated report of the experiment.


In [ ]:
best_head_epoch = int(np.argmin(history_head.history["val_loss"])) + 1
best_ft_epoch = int(np.argmin(history_fine_tune.history["val_loss"])) + 1

summary = f"""# Experiment: {EXP_NAME}

## Setup
- **Dataset**: {DATASET_NAME} ({DATASET_HANDLE})
- **Model**: {MODEL_NAME} (ImageNet pretrained)
- **Image size**: {IMAGE_SIZE}
- **Batch size**: {BATCH_SIZE}
- **Split protocol**: StratifiedGroupKFold by lesion_id (80/8/12% train/val/test)
- **Seed**: {SEED}
- **Head epochs**: {HEAD_EPOCHS} (LR={HEAD_LR})
- **Fine-tune epochs**: {FINE_TUNE_EPOCHS} (LR={FINE_TUNE_LR}, last {FINE_TUNE_LAYERS} layers)
- **Class weighting**: balanced
- **Augmentation**: RandomFlip, RandomRotation(0.1), RandomZoom(0.1), RandomContrast(0.1)

## Dataset
- **Total images**: {len(df):,}
- **Classes** ({num_classes}): {', '.join(class_names)}
- **Split sizes**: train={len(train_df):,}, validation={len(val_df):,}, test={len(test_df):,}

## Class Distribution
{class_distribution.to_string(index=False)}

## Class Weights
{weight_table.to_string(index=False)}

## Training
- **Stage 1 time**: {head_time:.1f}s
- **Stage 2 time**: {fine_tune_time:.1f}s
- **Total time**: {total_time:.1f}s
- **Best head epoch**: {best_head_epoch}
- **Best fine-tune epoch**: {best_ft_epoch}

## Test Results
- **Test accuracy**: {test_accuracy:.4f}
- **Macro F1**: {macro_f1:.4f}
- **Balanced accuracy**: {balanced_accuracy:.4f}
- **Macro ROC-AUC (OvR)**: {macro_roc_auc:.4f}

## Per-Class Metrics
{per_class_metrics.to_string(index=False)}

## Main Confusion Patterns
Top off-diagonal entries (>5):
"""

conf_copy = confusion.copy()
np.fill_diagonal(conf_copy, 0)
flat = [(i, j, conf_copy[i, j]) for i in range(num_classes) for j in range(num_classes) if conf_copy[i, j] > 5]
flat.sort(key=lambda x: -x[2])
for i, j, count in flat:
    summary += f"- {class_names[i]} misclassified as {class_names[j]}: {count} images\n"

summary += f"""
## Benchmark
- **Model parameters**: {model.count_params():,}
- **Model size**: {model_size_mb:.2f} MB
- **Median inference**: {median_ms_per_image:.2f} ms/image

---
Split protocol: Images sharing a lesion_id cannot appear in more than one split.
This is not a diagnostic system and must not be used for clinical decision-making.
"""

(Path(RESULTS_DIR) / "summary.md").write_text(summary)
print("\n✓ Summary saved")
print(f"\n✓ Experiment {EXP_NAME} complete. Results in {RESULTS_DIR}/")
